CrossNER bench

In [27]:
from dataloader import get_crossNER_data, data_process, ner_label_map, convert_ner_to_xml
from const import AI_CLASSS, SCIENCE_CLASS, LITERATURE_CLASSS, MUSIC_CLASSS, POLITICS_CLASSS
from utils import load_json, save_json

LABEL_DIC = {
    'ai': AI_CLASSS,
    'science': SCIENCE_CLASS,
    'literature': LITERATURE_CLASSS,
    'music': MUSIC_CLASSS,
    'politics': POLITICS_CLASSS,
}

# Load the CrossNER dataset
dataset = 'politics'
# subset = 'train'
# load datasets
train_sents = load_json(f'./datasets/crossner/{dataset}_train.json')
dev_sents = load_json(f'./datasets/crossner/{dataset}_dev.json')
label_map = LABEL_DIC[dataset]

# merge the train and dev sets
sents = train_sents + dev_sents
# processing one by one
new_data = []
for idx, sample in enumerate(sents):
    sentence, ners = data_process(sample)
    ners = ner_label_map(label_map, ners)
    response = convert_ner_to_xml(sample['str_words'], sample['tags_ner'], label_map)
    # append the new data
    new_data.append({
        'idx': idx,
        'text': sentence,
        'ners': ners,
        'target': response,
        'sent_id': sample['sent_id'],
        'str_words': sample['str_words'],
        'tags_ner': sample['tags_ner'],
    })
# save the new data to a json file
save_json(new_data, f'./datasets/zero_aug/{dataset}.json')

In [20]:
sample = sents[0]
print(sample.keys())

dict_keys(['str_words', 'tags_ner', 'tags_esi', 'tags_net', 'sent_id'])


In [21]:
sentence, ners = data_process(sample)
ners = ner_label_map(label_map, ners)
response = convert_ner_to_xml(sample['str_words'], sample['tags_ner'], label_map)

In [22]:
response

'Popular approaches of <entity type="product">opinion-based recommender system</entity> utilize various techniques including <entity type="field">text mining</entity> , <entity type="task">information retrieval</entity> , <entity type="task">sentiment analysis</entity> ( see also <entity type="task">Multimodal sentiment analysis</entity> ) and <entity type="field">deep learning</entity> <entity type="researcher">X.Y. Feng</entity> , <entity type="researcher">H. Zhang</entity> , <entity type="researcher">Y.J. Ren</entity> , <entity type="researcher">P.H. Shang</entity> , <entity type="researcher">Y. Zhu</entity> , <entity type="researcher">Y.C. Liang</entity> , <entity type="researcher">R.C. Guan</entity> , <entity type="researcher">D. Xu</entity> , ( 2019 ) , , 21 ( 5 ) : e12957 .'

MIT Resstant dataset

In [50]:
from openai import OpenAI

def llm_call_with_token_probs(prompt: str, system_prompt: str = "", model_name="deepseek-chat") -> str:
    """
    Calls the model with the given prompt and returns the response.

    Args:
        prompt (str): The user prompt to send to the model.
        system_prompt (str, optional): The system prompt to send to the model. Defaults to "".
        model (str, optional): The model to use for the call. Defaults to "claude-3-5-sonnet-20241022".

    Returns:
        str: The response from the language model.
        dict: The token probabilities from the model.
    """
    client = OpenAI(api_key="YOUR_API_KEY", base_url="https://api.deepseek.com") # TODO: Change the API key

    response = client.chat.completions.create(
        model=model_name, # deepseek-chat
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        max_tokens=4096,
        stream=False,
        logprobs=True,  # Get token probabilities
    )
    # return two parts of data
    # 1. The pure response
    ans = response.choices[0].message.content
    # 2. The token probabilities
    ans_token_logits = response.choices[0].logprobs.content
    return ans, ans_token_logits

In [51]:
test_str = "Temple University is"
response, logprobs = llm_call_with_token_probs(test_str, system_prompt="You are a helpful assistant.")

In [30]:
type(logprobs)

openai.types.chat.chat_completion.ChoiceLogprobs

In [55]:
logprobs[0]

ChatCompletionTokenLogprob(token='T', bytes=[84], logprob=-1.3113013e-06, top_logprobs=[])

In [57]:
logprobs[0].token

'T'

In [63]:
logprobs[-2]

ChatCompletionTokenLogprob(token=' life', bytes=[32, 108, 105, 102, 101], logprob=0.0, top_logprobs=[])

In [62]:
logprobs[-1]

ChatCompletionTokenLogprob(token=')?', bytes=[41, 63], logprob=0.0, top_logprobs=[])

In [64]:
new_logprobs = []
for i in range(len(logprobs)):
    new_logprobs.append({
        'token': logprobs[i].token,
        'bytes': logprobs[i].bytes,
        'logprob': logprobs[i].logprob,
    })

In [65]:
new_logprobs

[{'token': 'T', 'bytes': [84], 'logprob': -1.3113013e-06},
 {'token': 'emple', 'bytes': [101, 109, 112, 108, 101], 'logprob': 0.0},
 {'token': ' University',
  'bytes': [32, 85, 110, 105, 118, 101, 114, 115, 105, 116, 121],
  'logprob': 0.0},
 {'token': ' is', 'bytes': [32, 105, 115], 'logprob': 0.0},
 {'token': ' a', 'bytes': [32, 97], 'logprob': 0.0},
 {'token': ' public',
  'bytes': [32, 112, 117, 98, 108, 105, 99],
  'logprob': -1.192093e-07},
 {'token': ' research',
  'bytes': [32, 114, 101, 115, 101, 97, 114, 99, 104],
  'logprob': 0.0},
 {'token': ' university',
  'bytes': [32, 117, 110, 105, 118, 101, 114, 115, 105, 116, 121],
  'logprob': 0.0},
 {'token': ' located',
  'bytes': [32, 108, 111, 99, 97, 116, 101, 100],
  'logprob': 0.0},
 {'token': ' in', 'bytes': [32, 105, 110], 'logprob': 0.0},
 {'token': ' Philadelphia',
  'bytes': [32, 80, 104, 105, 108, 97, 100, 101, 108, 112, 104, 105, 97],
  'logprob': -4.5299635e-06},
 {'token': ',', 'bytes': [44], 'logprob': 0.0},
 {'tok